In [1]:
!pip install ludwig torch==2.0.1 torchtext==0.15.2 ptitprince bitsandbytes-cuda117 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.9 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.2/152.2 kB 8.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision

In [2]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn import metrics

import os
paths = []
for dirname, _, filenames in os.walk('/kaggle/input/data-ludwig-prueba/pruebas'):
    for filename in filenames:
        paths.append(os.path.join(dirname, filename))
        #print(os.path.join(dirname, filename))

In [3]:
df = pd.read_csv('/kaggle/input/data-ludwig-prueba/data_suborder_ludwig_relative.csv')

In [4]:
df['label'].value_counts()

label
1    250
0    250
Name: count, dtype: int64

In [5]:
df['audio_path'] = paths
df = df.sample(100)

In [6]:
# Verificar que df es un DataFrame válido
print("Tipo de df:", type(df))
print("Columnas de df:", df.columns)
print("Primeras filas de df:\n", df.head())

Tipo de df: <class 'pandas.core.frame.DataFrame'>
Columnas de df: Index(['label', 'audio_path'], dtype='object')
Primeras filas de df:
      label                                         audio_path
107      1  /kaggle/input/data-ludwig-prueba/pruebas/35389...
170      1  /kaggle/input/data-ludwig-prueba/pruebas/69395...
14       1  /kaggle/input/data-ludwig-prueba/pruebas/97139...
227      1  /kaggle/input/data-ludwig-prueba/pruebas/9229.mp3
295      0  /kaggle/input/data-ludwig-prueba/pruebas/14796...


In [7]:
df.sample(100)
output_path = '/kaggle/working/data_prueba.csv'
df.to_csv(output_path, index=False)

In [8]:
config_yaml = """
input_features:
    -
        name: audio_path
        type: audio
        encoder: 
            type: stacked_cnn
            reduce_output: concat
            conv_layers:
                -
                    num_filters: 16
                    filter_size: 6
                    pool_size: 4
                    pool_stride: 4
                    dropout: 0.2
                -
                    num_filters: 32
                    filter_size: 3
                    pool_size: 2
                    pool_stride: 2
                    dropout: 0.2
            fc_layers:
                -
                    output_size: 64
                    dropout: 0.2
        preprocessing:
            audio_feature:
                type: fbank
                window_length_in_s: 0.025
                window_shift_in_s: 0.01
                num_filter_bands: 80
            audio_file_length_limit_in_s: 1.0
            norm: per_file
            
output_features:
    -
        name: label
        type: binary

trainer:
    early_stop: 2
    epochs: 12
    batch_size: 4
    validation_metric: accuracy
"""

# Escribir la configuración a un archivo YAML
with open("config.yaml", "w") as f:
    f.write(config_yaml)

print("Configuración escrita en config.yml")

Configuración escrita en config.yml


In [9]:
from pydub import AudioSegment

def convert_to_mono(input_path, output_path):
    audio = AudioSegment.from_file(input_path)
    mono_audio = audio.set_channels(1)
    mono_audio.export(output_path, format="mp3")

def preprocess_audio_df(input_csv, output_csv, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    converted_paths = []

    for index, row in df.iterrows():
        original_path = row['audio_path']
        output_path = os.path.join(output_dir, os.path.basename(original_path))
        try:
            convert_to_mono(original_path, output_path)
            converted_paths.append(output_path)
        except Exception as e:
            print(f"Error converting {original_path}: {e}")
            converted_paths.append(None)

    df['audio_path'] = converted_paths
    df.to_csv(output_csv, index=False)

# Define paths
input_csv = '/kaggle/working/data_prueba.csv'
output_csv = '/kaggle/working/data_prueba_preprocessed.csv'
output_dir = '/kaggle/working/converted_audio_files'

In [10]:
preprocess_audio_df(input_csv, output_csv, output_dir)

In [11]:
pd.read_csv('/kaggle/working/data_prueba_preprocessed.csv').shape

(100, 2)

In [12]:
# Ahora puedes usar el nuevo CSV en Ludwig:
!ludwig train --dataset /kaggle/working/data_prueba_preprocessed.csv -c /kaggle/working/config.yaml --output_directory /kaggle/working/output

/opt/conda/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32
/opt/conda/lib/python3.10/site-packages/ludwig/data/preprocessing.py:1735: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dataset_cols[feature[COLUMN]] = dataset_cols[feature[COLUMN]].fillna(
/opt/conda/lib/python3.10/site-packages/ludwig/data/preprocessing.py:1735: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dataset_cols[feature[COLUMN]] = dataset_cols[feature[COLUMN]].fillna(
/opt/con

In [13]:
!ludwig evaluate --model_path /kaggle/working/output/experiment_run/model \
                 --dataset /kaggle/working/data_prueba_preprocessed.csv \
                 --split test \
                 --output_directory /kaggle/working/test_results

/opt/conda/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32
/opt/conda/lib/python3.10/site-packages/torch/nn/modules/conv.py:309: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ../aten/src/ATen/native/Convolution.cpp:1003.)
  return F.conv1d(input, weight, bias, self.stride,
/opt/conda/lib/python3.10/site-packages/ludwig/data/preprocessing.py:1735: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dataset_cols[feature[COLUMN]] = d

In [14]:
!ludwig visualize --visualization confusion_matrix \
                  --ground_truth_metadata /kaggle/working/output/experiment_run/model/training_set_metadata.json \
                  --test_statistics /kaggle/working/test_results/test_statistics.json \
                  --output_directory /kaggle/working/visualizations \
                  --file_format png

/opt/conda/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32
/opt/conda/lib/python3.10/site-packages/ludwig/utils/visualization_utils.py:1179: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels([""] + labels, rotation=45, ha="left")
/opt/conda/lib/python3.10/site-packages/ludwig/utils/visualization_utils.py:1180: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_yticklabels([""] + labels, rotation=45, ha="right")


In [15]:
from IPython import display
import ipywidgets
from pathlib import Path

ipywidgets.HBox([
  ipywidgets.Image(value=Path("/kaggle/working/visualizations/confusion_matrix_entropy__label_top2.png").read_bytes()),
  ipywidgets.Image(value=Path("/kaggle/working/visualizations/confusion_matrix__label_top2.png").read_bytes()),
])

In [16]:
import torchaudio
print(torchaudio.list_audio_backends())

['soundfile', 'sox_io']


In [17]:
#!ffmpeg -i /kaggle/input/data-ludwig-prueba/pruebas/793880.mp3 -ac 1 /kaggle/working/converted_793880.mp3

In [18]:
#!ludwig train --dataset /kaggle/working/data_prueba.csv -c /kaggle/working/config.yml --output_directory /kaggle/working/output